In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from tabpfn import TabPFNClassifier

# Leave-one-individual-out (LOIO) validation -- TabPFN

Same LOIO design as `08a_individual_validation_features_ml.ipynb` (see that
notebook for the full diagnostics, threshold rationale, and per-class fold
coverage check) -- this notebook loads the per-burst fold lookups built there
and runs **TabPFN** instead of Random Forest, so all three models are
compared on identical held-out sets.

Recap of scope:
- **Datasets**: Burst_1/uncorrected (raw-data baseline) and Burst_2/uncorrected
  (the final classifier's burst length).
  Bursts 2-4 are the same recordings as Burst_1 at finer child-window
  granularities, so each is run and reported separately, never pooled
  together.
- **Target**: behaviour only (the activity target is not used).
- An individual is a held-out test fold in a given burst only if they have
  >=30 total samples *in that burst*; smaller individuals stay in every
  fold's training set but are never the test fold. This is applied
  independently per burst -- Burst_2's eligible set (17/20) is larger than
  Burst_1's (13/20) since it has ~2x more rows per individual.
- **The behaviour model trains on the full label space -- no classes are
  dropped from training.** It sees all 8 behaviour classes, matching the
  main CV models in `05c_feature_classification_dl_tabpfn.ipynb`. Running,
  Self-scratching, and Sleeping are excluded only when computing pooled
  metrics (see the pooling cell), not from training -- see 08a for the full
  rationale.

As in `05c_feature_classification_dl_tabpfn.ipynb`, TabPFN has no fit-time
class/sample weighting (it does not train via gradient descent on the data);
`use_class_weight` here controls `balance_probabilities`, which rescales
`predict_proba`'s output by the training class frequencies at inference time.

## Load the per-burst fold lookups built in 08a

In [ ]:
BURSTS = ["burst_1", "burst_2"]
# Classes excluded from *pooled metrics only* (not from training -- see
# process_parquet_file and 08a's introduction for the full rationale).
DROPPED_MULTICLASS_CLASSES = ["Running", "Self-scratching", "Sleeping"]
TRAIN_ONLY_SENTINEL = "train_only"

fold_lookups = {}
per_burst_eligibility = {}
for burst in BURSTS:
    path = f"../data/temp/indices_loio_fold_lookup_{burst}.csv"
    fold_lookup = pd.read_csv(path)
    fold_lookups[burst] = fold_lookup

    eligible = sorted(
        fold_lookup.loc[fold_lookup["fold"] != TRAIN_ONLY_SENTINEL, "fold"]
        .unique()
        .tolist()
    )
    per_burst_eligibility[burst] = {"eligible": eligible}
    print(f"{burst}: eligible LOIO fold individuals ({len(eligible)}): {eligible}")

# The set of classes pooled multiclass metrics are restricted to -- the
# complement of DROPPED_MULTICLASS_CLASSES within whatever classes actually
# appear (unioned across bursts).
all_multiclass_classes = sorted(
    set().union(*[fold_lookups[b]["behaviour"].unique() for b in BURSTS])
)
RETAINED_MULTICLASS_CLASSES = [
    c for c in all_multiclass_classes if c not in DROPPED_MULTICLASS_CLASSES
]
print(
    f"Retained multiclass classes for metrics ({len(RETAINED_MULTICLASS_CLASSES)}): "
    f"{RETAINED_MULTICLASS_CLASSES}"
)

## TabPFN LOIO training/evaluation function

In [ ]:
def features_tabpfn_model(
    X_train,
    X_test,
    y_train,
    y_test,
    class_names,
    use_class_weight=True,
):
    """Train and evaluate TabPFN on a pre-split train/test pair (one LOIO fold).

    Mirrors features_tabpfn_model in 05c_feature_classification_dl_tabpfn.ipynb.
    """
    num_classes = len(class_names)
    model = TabPFNClassifier(random_state=42, balance_probabilities=use_class_weight)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)

        try:
            roc = roc_auc_score(
                y_test, y_pred_proba, multi_class="ovr", average="weighted"
            )
        except Exception:
            roc = np.nan

        metrics_results = [
            {
                "model": "TabPFN",
                "roc_auc": roc,
                "accuracy": model.score(X_test, y_test),
                "balanced_acc": balanced_accuracy_score(y_test, y_pred),
                "f1_score": f1_score(
                    y_test, y_pred, average="weighted", zero_division=0
                ),
                "precision": precision_score(
                    y_test, y_pred, average="weighted", zero_division=0
                ),
                "recall": recall_score(
                    y_test, y_pred, average="weighted", zero_division=0
                ),
                "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
                "precision_macro": precision_score(
                    y_test, y_pred, average="macro", zero_division=0
                ),
                "recall_macro": recall_score(
                    y_test, y_pred, average="macro", zero_division=0
                ),
                "time": time.time() - t0,
            }
        ]

        cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
        confusion_data = [
            {
                "model": "TabPFN",
                "actual_label": class_names[i],
                "predicted_label": class_names[j],
                "count": cm[i][j],
            }
            for i in range(num_classes)
            for j in range(num_classes)
        ]

    return pd.DataFrame(metrics_results), pd.DataFrame(confusion_data)

## LOIO orchestration for the behaviour target

In [ ]:
def process_parquet_file(
    file_path, fold_lookup, eligible_individuals, use_class_weight=True
):
    """Run LOIO cross-validation for the behaviour target on one file.

    The model trains on the full label space -- no classes are dropped here.
    It sees all 8 behaviour classes, matching the main CV models in
    05c_feature_classification_dl_tabpfn.ipynb. Sparse classes are excluded
    only when computing pooled metrics afterward (see the pooling cell), not
    from training. Identical structure to 08a's process_parquet_file -- only
    the per-fold trainer (features_tabpfn_model vs. features_ml_models)
    differs.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    m = fold_lookup[["Ind_ID", "new_burst", "behaviour", "activity", "fold"]]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner")

    X_full = df.drop(
        ["Ind_ID", "new_burst", "behaviour", "activity", "fold"],
        axis=1,
        errors="ignore",
    )

    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    metrics, confs = [], []
    for held_out in eligible_individuals:
        tr = (df["fold"] != held_out).to_numpy()
        te = (df["fold"] == held_out).to_numpy()
        if te.sum() == 0:
            continue
        met, conf = features_tabpfn_model(
            X_full[tr],
            X_full[te],
            y_enc[tr],
            y_enc[te],
            class_names,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["held_out_individual"] = held_out
            d["n_test_samples"] = int(te.sum())
        metrics.append(met)
        confs.append(conf)

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "held_out_individual",
        "n_test_samples",
        "model",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "held_out_individual",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: pd.concat(lst)[cols] if lst else pd.DataFrame(columns=cols)
    return {
        "multiclass_metrics": fin(metrics, metric_columns),
        "multiclass_conf": fin(confs, conf_columns),
    }

## Run LOIO across datasets

In [ ]:
DATASETS = [
    ("burst_1", "uncorrected"),
    ("burst_2", "uncorrected"),
]

all_results = []
for burst, correction_type in DATASETS:
    fold_lookup = fold_lookups[burst]
    eligible = per_burst_eligibility[burst]["eligible"]
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file(pq_file_path, fold_lookup, eligible)
    all_results.append(result)

multiclass_metrics = pd.concat(
    [r["multiclass_metrics"] for r in all_results], ignore_index=True
)
multiclass_conf = pd.concat(
    [r["multiclass_conf"] for r in all_results], ignore_index=True
)

## Pool results and inspect per-individual breakdown

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name, retained_classes=None):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-individual metrics when many individuals have zero test examples of
    some classes (e.g. grooming for individuals who never groom).

    Args:
        retained_classes (list, optional): If given, metrics are computed
            only over rows whose TRUE label is one of these classes, keeping
            whatever the model actually predicted (even a class outside this
            set counts as a genuine miss). Training itself is never
            restricted -- see process_parquet_file -- this only controls
            which true-label rows count toward the reported metrics, so the
            model stays an 8-class classifier and only the comparison metric
            is restricted to the well-populated classes.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())

    if retained_classes is not None:
        mask = np.isin(y_true, retained_classes)
        y_true, y_pred = y_true[mask], y_pred[mask]
        labels_kwarg = {"labels": retained_classes}
    else:
        labels_kwarg = {}

    if len(y_true) == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}

    # balanced_accuracy_score has no `labels` param, so when restricting to a
    # subset of classes we use macro recall instead -- mathematically the
    # same thing balanced accuracy is (mean per-class recall), and this
    # matches the convention already used elsewhere (recall_macro ==
    # balanced_acc throughout 05a-05d's own metrics).
    balanced_acc = (
        recall_score(y_true, y_pred, average="macro", zero_division=0, **labels_kwarg)
        if retained_classes is not None
        else balanced_accuracy_score(y_true, y_pred)
    )

    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_acc,
        "f1_score": f1_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "recall": recall_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
    }


def pooled_summary_per_dataset(conf_df, model_name, retained_classes=None):
    """Pooled metrics computed separately per (burst, correction_type), so
    datasets are never mixed into a single pool (Burst_2 is the same
    recordings as Burst_1 at finer windowing -- pooling across bursts would
    leak near-duplicate windows across train/test)."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(
            grp, model_name, retained_classes=retained_classes
        )
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


pooled_multiclass_summary = pooled_summary_per_dataset(
    multiclass_conf, "TabPFN", retained_classes=RETAINED_MULTICLASS_CLASSES
)
print("Pooled multiclass LOIO metrics per dataset (restricted to retained classes):")
print(pooled_multiclass_summary)

# Per-individual breakdown, alongside their sample counts, for transparency --
# this is the honest complement to the pooled summary: it shows how much the
# pooled result is driven by well-sampled vs. sparse individuals. Note these
# per-individual accuracy/balanced_acc/f1_macro values come straight from
# features_tabpfn_model and are NOT restricted to RETAINED_MULTICLASS_CLASSES
# (that restriction is only applied in the pooled summary above) -- they
# reflect genuine 8-class performance for that held-out individual.
per_individual_cols = [
    "burst",
    "correction_type",
    "held_out_individual",
    "n_test_samples",
    "model",
    "accuracy",
    "balanced_acc",
    "f1_macro",
]
per_individual_multiclass = multiclass_metrics[per_individual_cols].sort_values(
    ["burst", "correction_type", "n_test_samples"], ascending=[True, True, False]
)
print("\nPer-individual multiclass results (unrestricted, all 8 classes):")
print(per_individual_multiclass)

## Save outputs

In [ ]:
multiclass_metrics.to_csv(
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_loio_metrics.csv",
    index=False,
)
multiclass_conf.to_csv(
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_loio_confusion.csv",
    index=False,
)
pooled_multiclass_summary.to_csv(
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_loio_pooled_summary.csv",
    index=False,
)